<a href="https://colab.research.google.com/github/di-claro-batera/faculdade_data_science/blob/main/Desafio_rede_bin%C3%A1ria_de_Hopfield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [4]:
class HopfieldNetwork:
  """
  Implementação de uma Rede Neural de Hopfield.
  """
  def __init__(self, num_neurons):
    """
    Inicializa a Rede Neural.

    Args:
      num_neurons: Número de neurônios na Rede Neural.
    """
    self.num_neurons = num_neurons
    # Inicializa a matriz de pesos com zeros
    self.weights = np.zeros((num_neurons, num_neurons))

  def train(self, patterns):
    """
    Treina a rede usando a Regra de Hebb para armazenar os padrões.

    Args:
      patterns (list of np.array): Uma Lista de padrões a serem armazenados. Os padrões devem estar em formato bipolar.
    """
    num_patterns = len(patterns)
    print("Iniciando o treinamento com {} padrões.".format(num_patterns))

    # Aplicando a Regra de Hebb
    for p in patterns:
      # Garante que o padrão é um vetor de coluna para o produto externo
      p = p.reshape(-1, 1)
      self.weights += p @ p.T # Produto externo: p * p_transposto

    # Zera a diagonal principal para evitar auto-conexões
    np.fill_diagonal(self.weights, 0)

    print("Treinamento concluído.")

  def recall(self, pattern, max_iter=10):
    """
    Tenta recuperar um padrão memorizado a partir de um padrão de entrada.

    Args:
      pattern (np.array): O padrão de entrada (potencialmente ruidoso).
      max_iter (int): O número máximo de iterações para a convergência.

    Returns:
      np.array: O padrão recuperado após a convergência ou max_iter.
    """

    current_pattern = pattern.copy()
    print(f"\nPadrão de entrada para recuperação: {current_pattern}")

    for i in range(max_iter):
      print(f"--- Iteração {i+1} ---")

      # Atualização assíncrona dos neurônios
      for j in range(self.num_neurons):
        # Calcula o produto escalar entre os pesos e o padrão atual
        dot_product = np.dot(self.weights[j, :], current_pattern)
        # Função de ativação (limiar em 0)
        current_pattern[j] = 1 if dot_product > 0 else -1
      print(f"Padrão atualizado: \n{current_pattern}")

      # Verifica se o padrão estabilizou
      if np.array_equal(current_pattern, self.recall_step(current_pattern)):
        print("\nA rede convergiu para um padrão estável.")
        break

    return current_pattern

  def recall_step(self, pattern):
    """Helper para verificar a convergência."""
    new_pattern = pattern.copy()
    for j in range(self.num_neurons):
      dot_product = np.dot(self.weights[j, :], new_pattern)
      new_pattern[j] = 1 if dot_product > 0 else -1
    return new_pattern

def binary_to_bipolar(pattern):
  """Converte um padrão binário (0,1) para bipolar (-1,1)."""
  return np.array([1 if x == 1 else -1 for x in pattern])

In [5]:
# --- PROGRAMA PRINCIPAL ---

# 1. Definir os padrões binários
pattern_01_bin = [0, 0, 0, 0, 0, 0, 1, 1, 1]
pattern_02_bin = [0, 0, 0, 1, 1, 1, 0, 0, 0]
pattern_03_bin = [1, 1, 1, 0, 0, 0, 0, 0, 0]

# 2. Converter para o formato bipolar
p1 = binary_to_bipolar(pattern_01_bin)
p2 = binary_to_bipolar(pattern_02_bin)
p3 = binary_to_bipolar(pattern_03_bin)

patterns_to_store = [p1, p2, p3]

# 3. Criar e treinar a rede de Hopfield
num_neurons = 9
hopfield_net = HopfieldNetwork(num_neurons)
hopfield_net.train(patterns_to_store)

# 4. Exibir a matriz de pesos final
print("\n--- Matriz de Pesos (W) Estabelecida ---")
#Configura a impressão para melhor visualização
np.set_printoptions(linewidth=200, formatter={'float': '{:.0f}'.format})
print(hopfield_net.weights)

# 5. Teste de Recuperação (opcional, mas recomendado)
# Criar um padrão com ruidos (Padrão 03 com um bit invertido)
noisy_pattern = np.array([1, 1, 1, -1, 1, -1, -1, -1, -1]) # bit 5 and 9 inverted

# Tentar recuperar o padrão original
recovered_pattern = hopfield_net.recall(noisy_pattern)

print("\n--- Resultado da Recuperação ---")
print(f"Padrão Original (P3):\n{p3}")
print(f"Padrão Recuperado: \n{recovered_pattern}")

# Verificar se a recuperação foi bem sucedida
if np.array_equal(p3, recovered_pattern):
  print("\nSucesso! O padrão original foi recuperado corretamente.")
else:
  print("\nFalha. A rede convergiu para um estado espúrio ou outro padrão.")

Iniciando o treinamento com 3 padrões.
Treinamento concluído.

--- Matriz de Pesos (W) Estabelecida ---
[[0 3 3 -1 -1 -1 -1 -1 -1]
 [3 0 3 -1 -1 -1 -1 -1 -1]
 [3 3 0 -1 -1 -1 -1 -1 -1]
 [-1 -1 -1 0 3 3 -1 -1 -1]
 [-1 -1 -1 3 0 3 -1 -1 -1]
 [-1 -1 -1 3 3 0 -1 -1 -1]
 [-1 -1 -1 -1 -1 -1 0 3 3]
 [-1 -1 -1 -1 -1 -1 3 0 3]
 [-1 -1 -1 -1 -1 -1 3 3 0]]

Padrão de entrada para recuperação: [ 1  1  1 -1  1 -1 -1 -1 -1]
--- Iteração 1 ---
Padrão atualizado: 
[ 1  1  1 -1 -1 -1 -1 -1 -1]

A rede convergiu para um padrão estável.

--- Resultado da Recuperação ---
Padrão Original (P3):
[ 1  1  1 -1 -1 -1 -1 -1 -1]
Padrão Recuperado: 
[ 1  1  1 -1 -1 -1 -1 -1 -1]

Sucesso! O padrão original foi recuperado corretamente.
